In [ ]:
from pathlib import Path
import copy
import json
import logging
import os
import re
import shutil
import time

import jsonpickle
import matplotlib.pyplot as plt
import napari
import numpy as np
import pandas as pd
import seaborn as sns
import tifffile
from scipy.ndimage import binary_dilation, shift as ndi_shift
from skimage.segmentation import expand_labels

import flofish
import flofish.utils
from flofish.experiment import Experiment
from flofish.image import Image

CONFIG_FILE = Path(os.environ.get("FISH_IF_CONFIG", "config.json")).expanduser().resolve()
if not CONFIG_FILE.is_file():
    raise FileNotFoundError(f"Config not found: {CONFIG_FILE}")
with CONFIG_FILE.open(encoding="utf-8") as handle:
    CONFIG = json.load(handle)


def resolve_config_path(value):
    path = Path(value).expanduser()
    return path.resolve() if path.is_absolute() else (CONFIG_FILE.parent / path).resolve()


INPUT_DIR = resolve_config_path(CONFIG["inputdir"])
OUTPUT_ROOT = resolve_config_path(CONFIG["outputdir"])
ANALYSIS = CONFIG.get("analysis", {})
ROLES = ANALYSIS.get("roles", {})
for role in ("mrna", "protein", "nuclear"):
    if role not in ROLES:
        raise ValueError(f"Missing analysis.roles.{role} in config.json")
    for key in ("channel", "image_file"):
        if key not in ROLES[role]:
            raise ValueError(f"Missing analysis.roles.{role}.{key}")
    if ROLES[role]["channel"] not in CONFIG.get("channels", {}):
        raise ValueError(f"Role {role!r} points to an unconfigured channel")

MRNA_ROLE = ROLES["mrna"]
PROTEIN_ROLE = ROLES["protein"]
NUCLEAR_ROLE = ROLES["nuclear"]
MRNA_DATA_KEY = MRNA_ROLE.get(
    "data_key", CONFIG["channels"][MRNA_ROLE["channel"]].get("mrna", MRNA_ROLE["channel"])
)
DIC_IMAGE_FILE = ANALYSIS.get("dic_image_file", "DIC.tif")
EXPECTED_IMAGE_COUNT = ANALYSIS.get("expected_image_count")
SPOT_SOURCE = ANALYSIS.get("spot_source", "decomposed_spots")
if SPOT_SOURCE != "decomposed_spots":
    raise ValueError("This pipeline requires Gaussian-decomposed spots; set spot_source to decomposed_spots")
DAPI_MIN_OVERLAP_PX = int(ANALYSIS.get("dapi_min_overlap_px", 10))
MASK_EXPANSION_PX = float(ANALYSIS.get("mask_expansion_px", 1))
BACKGROUND_EXCLUSION_PX = int(ANALYSIS.get("protein_background_exclusion_px", 3))
TIMEPOINTS = ANALYSIS.get("timepoints", [0, 20, 40, 60])
if DAPI_MIN_OVERLAP_PX < 1 or MASK_EXPANSION_PX < 0 or BACKGROUND_EXCLUSION_PX < 0:
    raise ValueError("Invalid overlap, expansion, or background-exclusion setting")

INITIAL_CELL_MASK = "DIC_masks.tif"
RAW_CELL_MASK = "DIC_masks_unshifted.tif"
CORRECTED_CELL_MASK = "DIC_masks_pp_corrected.tif"
FLOFISH_CELL_MASK = "DIC_masks_pp.tif"
EXPANDED_CELL_MASK = "DIC_masks_pp_expanded.tif"
DAPI_MASK = "DAPI_masks.tif"
NUCLEAR_FILTERED_MASK = "DIC_masks_pp_corrected_nuclear_filtered.tif"
STAGE04_FILE = "img.04.pkl"
MASTER_CSV = OUTPUT_ROOT / "master_mrna_protein_decomposed.csv"
IMAGE_QC_CSV = OUTPUT_ROOT / "image_level_qc_decomposed.csv"
PLOT_DIR = OUTPUT_ROOT / "quantification_plots_decomposed"
if not INPUT_DIR.is_dir():
    raise FileNotFoundError(f"Input directory not found: {INPUT_DIR}")
print(f"Config: {CONFIG_FILE}\nInput: {INPUT_DIR}\nOutput: {OUTPUT_ROOT}")
print(f"mRNA: {MRNA_ROLE['channel']} (data key: {MRNA_DATA_KEY})")
print(f"Protein-associated fluorescence: {PROTEIN_ROLE['channel']}")

In [ ]:
# %% 0B. Shared helpers

def current_analysis():
    """Reload review settings without rerunning the main configuration cell."""
    with CONFIG_FILE.open(encoding="utf-8") as handle:
        return json.load(handle).get("analysis", {})


def flofish_experiment():
    """Create a FloFish Experiment using the original configuration file."""
    if not CONFIG_FILE.is_file():
        raise FileNotFoundError(f"Configuration file not found: {CONFIG_FILE}")

    exp = Experiment.from_cfg_file(str(CONFIG_FILE))
    exp.inputdir = str(INPUT_DIR)
    exp.outputdir = str(OUTPUT_ROOT)
    return exp


def image_folders():
    if not OUTPUT_ROOT.is_dir():
        raise FileNotFoundError(OUTPUT_ROOT)
    folders = sorted(p for p in OUTPUT_ROOT.iterdir() if p.is_dir() and (p / "img.json").is_file())
    if not folders:
        raise RuntimeError(f"No FloFish image folders in {OUTPUT_ROOT}")
    if EXPECTED_IMAGE_COUNT is not None and len(folders) != int(EXPECTED_IMAGE_COUNT):
        raise RuntimeError(f"Expected {EXPECTED_IMAGE_COUNT} images; found {len(folders)}")
    return folders


def parse_metadata(folder_name):
    settings = ANALYSIS.get("metadata", {})
    override = settings.get("by_folder", {}).get(folder_name, {})
    strain = override.get("strain", settings.get("default_strain"))
    timepoint = override.get("timepoint_min", settings.get("default_timepoint_min"))
    strain_pattern = settings.get("strain_regex")
    time_pattern = settings.get("timepoint_regex")
    if "strain" not in override and strain_pattern:
        match = re.search(strain_pattern, folder_name, flags=re.IGNORECASE)
        if match:
            strain = match.group(1)
    if "timepoint_min" not in override and time_pattern:
        match = re.search(time_pattern, folder_name, flags=re.IGNORECASE)
        if match:
            timepoint = float(match.group(1))
    if settings.get("required", False) and (strain is None or timepoint is None):
        raise ValueError(f"Missing strain/time metadata for {folder_name}; update analysis.metadata")
    return strain if strain is not None else "unknown", timepoint


def read_label_mask(path):
    mask = np.asarray(tifffile.imread(path))
    if mask.ndim != 2 or not np.issubdtype(mask.dtype, np.integer):
        raise ValueError(f"Expected a 2D integer label mask: {path}; got {mask.shape}, {mask.dtype}")
    if np.any(mask < 0):
        raise ValueError(f"Negative labels in {path}")
    return mask


def display_projection(path):
    image = np.asarray(tifffile.imread(path))
    if image.ndim == 2:
        return image.astype(float)
    if image.ndim == 3:
        return image.max(axis=0).astype(float)
    raise ValueError(f"Expected Y/X or Z/Y/X TIFF: {path}; got {image.shape}")


def load_stage04(folder):
    with (folder / STAGE04_FILE).open(encoding="utf-8") as handle:
        return jsonpickle.decode(handle.read())


def mrna_spots(image):
    mrna = getattr(image, "mrna", None)
    if not isinstance(mrna, dict) or MRNA_DATA_KEY not in mrna:
        keys = list(mrna) if isinstance(mrna, dict) else []
        raise KeyError(f"mRNA data key {MRNA_DATA_KEY!r} missing; available: {keys}")
    data = mrna[MRNA_DATA_KEY]
    if not isinstance(data, dict) or SPOT_SOURCE not in data or data[SPOT_SOURCE] is None:
        raise ValueError(f"Required {SPOT_SOURCE!r} result is missing for {MRNA_DATA_KEY}")
    spots = np.asarray(data[SPOT_SOURCE])
    if spots.size == 0:
        return np.empty((0, 3), dtype=float)
    if spots.ndim != 2 or spots.shape[1] < 3:
        raise ValueError(f"Expected spot coordinates in Z/Y/X order; got {spots.shape}")
    if not np.isfinite(spots[:, :3]).all():
        raise ValueError("Non-finite mRNA spot coordinates")
    return spots


def assign_spots(spots, mask):
    """Retain the original notebook's integer-truncation Y/X convention."""
    labels = np.zeros(len(spots), dtype=np.int64)
    if len(spots):
        ys, xs = spots[:, 1].astype(int), spots[:, 2].astype(int)
        valid = (ys >= 0) & (ys < mask.shape[0]) & (xs >= 0) & (xs < mask.shape[1])
        labels[valid] = mask[ys[valid], xs[valid]]
    return labels


def touches_border(mask, label):
    return bool(np.any(mask[0] == label) or np.any(mask[-1] == label)
                or np.any(mask[:, 0] == label) or np.any(mask[:, -1] == label))

In [ ]:
# %% 1. Configure images and export aligned input layers — fresh output only
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
if any(OUTPUT_ROOT.iterdir()):
    raise RuntimeError(f"Use an empty output directory for Stage 1: {OUTPUT_ROOT}")
exp = flofish_experiment()
exp.create_image_list()
if not exp.images:
    raise RuntimeError("FloFish found no images; check inputdir, experiments, and channels")
if EXPECTED_IMAGE_COUNT is not None and len(exp.images) != int(EXPECTED_IMAGE_COUNT):
    raise RuntimeError(f"Expected {EXPECTED_IMAGE_COUNT} images; found {len(exp.images)}")
for index, parameters in enumerate(exp.images.values(), start=1):
    print(f"[{index}/{len(exp.images)}] Configuring and aligning")
    image = Image.from_dict(parameters, exp)
    tic = time.time()
    image.read_image()
    image.read_cells()
    image.align()
    image.create_grgb()
    image.save_input_layers()
    image.time["01-configure"] = time.time() - tic
    image.save_metadata("configure")
for folder in image_folders():
    for role in ROLES.values():
        path = folder / role["image_file"]
        if not path.is_file():
            raise FileNotFoundError(
                f"Configured role TIFF missing: {path}. Check actual exported filenames "
                "and analysis.roles; do not assume channel keys equal TIFF stems."
            )
print("Stage 1 complete")

In [ ]:
# %% 2. DIC-cell and DAPI-nucleus segmentation
# The underlying FloFish segment_dapi() API is intentionally unchanged.
exp = flofish_experiment()
exp.read_image_list_from_jsons()
if not exp.json_files:
    raise RuntimeError("No img.json files; run Stage 1")
exp.init_omnipose()
for index, image_json in enumerate(exp.json_files, start=1):
    print(f"[{index}/{len(exp.json_files)}] Segmenting {Path(image_json).parent.name}")
    image = Image.from_json(image_json, exp)
    tic = time.time()
    image.segment_cells()
    image.time["02-segment-cells"] = time.time() - tic
    tic = time.time()
    image.segment_dapi()
    image.time["02-segment-dapi"] = time.time() - tic
    image.save("02")
    image.save_metadata("segment")
print("Stage 2 complete")

In [ ]:
# %% 3. Manual mask-alignment helpers

def ensure_raw_mask(folder):
    """Preserve the initial mask once; all shifts start from this copy."""
    folder = Path(folder)
    raw = folder / RAW_CELL_MASK
    if not raw.is_file():
        source = folder / INITIAL_CELL_MASK
        if not source.is_file():
            raise FileNotFoundError(source)
        shutil.copyfile(source, raw)
    return raw


def shifted_mask(folder, dy=0, dx=0):
    """Positive dy moves down; positive dx moves right. Units: pixels."""
    if not np.isfinite([dy, dx]).all():
        raise ValueError("Mask shifts must be finite numbers")
    raw = read_label_mask(ensure_raw_mask(folder))
    return ndi_shift(
        raw, shift=(dy, dx), order=0, mode="constant", cval=0,
        prefilter=False,
    ).astype(raw.dtype)


def preview_mask_shift(folder, dy=0, dx=0):
    """Display a proposed shift without writing a corrected mask."""
    folder = Path(folder)
    viewer = napari.Viewer()
    dic_path = folder / DIC_IMAGE_FILE
    if dic_path.is_file():
        viewer.add_image(display_projection(dic_path), name="DIC", colormap="gray")
    for role_name, role in ROLES.items():
        path = folder / role["image_file"]
        if path.is_file():
            colour = role.get(
                "colormap",
                CONFIG["channels"][role["channel"]].get("colormap", "gray"),
            )
            viewer.add_image(
                display_projection(path),
                name=f"{role_name}: {role['channel']}",
                colormap=colour, opacity=0.55,
                visible=role_name == "nuclear",
            )
    viewer.add_labels(
        shifted_mask(folder, dy, dx),
        name=f"Proposed cells: dy={dy}, dx={dx}", opacity=0.35,
    )
    return viewer


def save_mask_shift(folder, dy=0, dx=0):
    """Save the corrected mask and replace this field's entry in the shift log."""
    folder = Path(folder)
    target = folder / CORRECTED_CELL_MASK
    tifffile.imwrite(target, shifted_mask(folder, dy, dx))
    log_path = OUTPUT_ROOT / "manual_mask_shifts.csv"
    row = pd.DataFrame([{
        "Image_Folder": folder.name,
        "dy_px": float(dy), "dx_px": float(dx),
        "final_mask_file": CORRECTED_CELL_MASK,
    }])
    old = pd.read_csv(log_path) if log_path.is_file() else pd.DataFrame(columns=row.columns)
    if "Image_Folder" not in old.columns:
        raise ValueError("Existing shift log uses a different schema; archive it before continuing")
    old = old.loc[old["Image_Folder"] != folder.name]
    pd.concat([old, row], ignore_index=True).sort_values("Image_Folder").to_csv(
        log_path, index=False,
    )
    print(f"Saved {target}\nShift: dy={dy} px, dx={dx} px")


In [ ]:
# %% 3B. Manual review — edit these values IN THE NOTEBOOK
# First run with SAVE_FINAL_SHIFT=False, inspect Napari, and adjust DY_PX/DX_PX.
# After confirming the proposed shift, set SAVE_FINAL_SHIFT=True and rerun.
# Reset SAVE_FINAL_SHIFT=False before selecting another field.
# This cell previews every time; when True it also saves immediately.

IMAGE_INDEX = 0
DY_PX = 0
DX_PX = 0
SAVE_FINAL_SHIFT = False

folders = image_folders()
if not 0 <= IMAGE_INDEX < len(folders):
    raise IndexError(f"IMAGE_INDEX must be between 0 and {len(folders) - 1}")
folder = folders[IMAGE_INDEX]
print(f"Reviewing [{IMAGE_INDEX}/{len(folders) - 1}] {folder.name}")
print(f"Proposed shift: dy={DY_PX} px, dx={DX_PX} px")
viewer = preview_mask_shift(folder, dy=DY_PX, dx=DX_PX)
if SAVE_FINAL_SHIFT:
    save_mask_shift(folder, dy=DY_PX, dx=DX_PX)
else:
    print("Preview only. Confirm alignment, then set SAVE_FINAL_SHIFT=True and rerun.")



In [ ]:
# %% 3C. Finalise corrected masks — run AFTER reviewing all fields
# Saved corrections are preserved. Fields without a saved correction receive
# the original unshifted mask. This does NOT automatically review those fields.
# Set ALL_FIELDS_REVIEWED=True only after confirming every field's alignment.
# Do not use the previous config-driven Stage 3C after using this replacement.

ALL_FIELDS_REVIEWED = False

if not ALL_FIELDS_REVIEWED:
    raise RuntimeError("Review all fields first, then set ALL_FIELDS_REVIEWED=True in this cell")

folders = image_folders()
log_path = OUTPUT_ROOT / "manual_mask_shifts.csv"
columns = ["Image_Folder", "dy_px", "dx_px", "final_mask_file"]
log = pd.read_csv(log_path) if log_path.is_file() else pd.DataFrame(columns=columns)
if not set(columns).issubset(log.columns):
    raise ValueError("Existing shift log uses a different schema; archive it before continuing")

for folder in folders:
    target = folder / CORRECTED_CELL_MASK
    if target.is_file():
        # Do not overwrite previously saved shifts or guess their offsets.
        if folder.name not in set(log["Image_Folder"]):
            raise RuntimeError(
                f"Corrected mask exists without a shift-log entry: {folder.name}. "
                "Review it and save its confirmed shift using Stage 3B."
            )
        read_label_mask(target)
        continue
    tifffile.imwrite(target, shifted_mask(folder, dy=0, dx=0))
    log = log.loc[log["Image_Folder"] != folder.name]
    log = pd.concat([log, pd.DataFrame([{
        "Image_Folder": folder.name, "dy_px": 0.0, "dx_px": 0.0,
        "final_mask_file": CORRECTED_CELL_MASK,
    }])], ignore_index=True)

log.sort_values("Image_Folder").to_csv(log_path, index=False)
print(f"Corrected masks finalised for {len(folders)} fields")
print(f"Shift log: {log_path}")
print("Continue with Stage 4 of the config-driven pipeline.")

In [ ]:
# %% 4. Create FloFish-compatible corrected and expanded masks
for folder in image_folders():
    mask = read_label_mask(folder / CORRECTED_CELL_MASK)
    tifffile.imwrite(folder / FLOFISH_CELL_MASK, mask)
    tifffile.imwrite(folder / EXPANDED_CELL_MASK,
                    expand_labels(mask, distance=MASK_EXPANSION_PX).astype(mask.dtype))
print("Stage 4 complete")

In [ ]:
# %% 5. Selected-mRNA-channel detection and Gaussian decomposition
if not hasattr(flofish.utils, "original_count_spots"):
    flofish.utils.original_count_spots = flofish.utils.count_spots
_original_count_spots = flofish.utils.original_count_spots


def safe_count_spots(mask, nuclear_mask, spot_data, cells):
    return _original_count_spots(mask, nuclear_mask, np.asarray(spot_data).astype(int), cells)


def restrict_to_mrna_channel(image):
    original_load = image.load_data

    def selected_channel_load(data_type, *args, **kwargs):
        result = original_load(data_type, *args, **kwargs)
        if data_type == "mrna":
            if MRNA_DATA_KEY not in image.mrna:
                raise KeyError(f"Required mRNA key {MRNA_DATA_KEY!r}; available: {list(image.mrna)}")
            image.mrna = {MRNA_DATA_KEY: image.mrna[MRNA_DATA_KEY]}
        return result

    image.load_data = selected_channel_load
    return original_load


exp = flofish_experiment()
exp.read_image_list_from_jsons()
if not exp.json_files:
    raise RuntimeError("No img.json files")
completed, failed = [], []
flofish.utils.count_spots = safe_count_spots
try:
    for index, image_json in enumerate(exp.json_files, start=1):
        folder_name = Path(image_json).parent.name
        print(f"[{index}/{len(exp.json_files)}] mRNA detection: {folder_name}", flush=True)
        try:
            image = Image.from_json(image_json, exp)
            image.read_image()
            original_load = restrict_to_mrna_channel(image)
            try:
                image.find_focus()
                image.filter()
                image.detect_spots()
                image.decompose_spots()
                mrna_spots(image)  # Require decomposition output; no silent fallback.
            finally:
                image.load_data = original_load
            image.save("04")
            image.save_metadata("spot_detection")
            completed.append(folder_name)
        except Exception as error:
            logging.exception("mRNA detection/decomposition failed for %s", folder_name)
            failed.append(f"{folder_name}: {error}")
finally:
    flofish.utils.count_spots = _original_count_spots
if failed:
    raise RuntimeError("mRNA detection failed; do not compile partial results:\n - " + "\n - ".join(failed))
print(f"Stage 5 complete: {len(completed)} images")

In [ ]:
# %% 5B. Filter corrected cell masks by nuclear overlap
per_cell_qc, per_image_qc = [], []
for folder in image_folders():
    cells = read_label_mask(folder / CORRECTED_CELL_MASK)
    nuclei = read_label_mask(folder / DAPI_MASK)
    if cells.shape != nuclei.shape:
        raise ValueError(f"Cell/nuclear mask shape mismatch in {folder.name}")
    labels = np.unique(cells)
    labels = labels[labels > 0]
    retained = np.zeros_like(cells)
    retained_n = 0
    for label in labels:
        region = cells == label
        overlap = int(np.count_nonzero(region & (nuclei > 0)))
        keep = overlap >= DAPI_MIN_OVERLAP_PX
        if keep:
            retained[region] = label
            retained_n += 1
        per_cell_qc.append({"Image_Folder": folder.name, "cell_id": int(label),
                            "Nuclear_Overlap_px": overlap, "Retained": keep})
    tifffile.imwrite(folder / NUCLEAR_FILTERED_MASK, retained)
    per_image_qc.append({"Image_Folder": folder.name, "N_cells_before": len(labels),
                         "N_cells_retained": retained_n,
                         "Retention_fraction": retained_n / len(labels) if len(labels) else np.nan})
pd.DataFrame(per_cell_qc).to_csv(OUTPUT_ROOT / "nuclear_filter_per_cell_qc.csv", index=False)
pd.DataFrame(per_image_qc).to_csv(OUTPUT_ROOT / "nuclear_filter_per_image_qc.csv", index=False)
print("Nuclear filtering complete")


In [ ]:
# %% 6. Compile per-cell mRNA and protein fluorescence WITHOUT strain metadata

def parse_timepoint(folder_name):
    """Read time from an exact-folder override, regex, or optional default."""
    settings = ANALYSIS.get("metadata", {})
    override = settings.get("by_folder", {}).get(folder_name, {})
    if "timepoint_min" in override:
        value = override["timepoint_min"]
    else:
        value = settings.get("default_timepoint_min")
        pattern = settings.get("timepoint_regex")
        if pattern:
            match = re.search(pattern, folder_name, flags=re.IGNORECASE)
            if match:
                if match.lastindex is None:
                    raise ValueError("timepoint_regex must contain a capturing group for the time")
                value = match.group(1)
    if value is None:
        if settings.get("required", False):
            raise ValueError(
                f"Missing timepoint for {folder_name}. Add an explicit time in "
                "analysis.metadata.by_folder or correct timepoint_regex."
            )
        return np.nan
    value = float(value)
    if not np.isfinite(value) or value < 0:
        raise ValueError(f"Invalid timepoint for {folder_name}: {value}")
    return value


MASTER_CSV = OUTPUT_ROOT / "master_per_cell_mrna_protein_decomposed.csv"
IMAGE_QC_CSV = OUTPUT_ROOT / "image_level_mrna_protein_qc_decomposed.csv"
PLOT_DIR = OUTPUT_ROOT / "strain_free_quantification_plots"

folders = image_folders()
required = [
    STAGE04_FILE, NUCLEAR_FILTERED_MASK,
    CORRECTED_CELL_MASK, PROTEIN_ROLE["image_file"],
]
missing = [
    f"{folder.name}/{name}"
    for folder in folders for name in required
    if not (folder / name).is_file()
]
if missing:
    raise FileNotFoundError("Missing compilation inputs:\n - " + "\n - ".join(missing))
if MASTER_CSV.exists() or IMAGE_QC_CSV.exists():
    raise FileExistsError(
        "Final CSV already exists. Archive/remove the strain-free CSVs only "
        "when deliberately recompiling."
    )

all_cells, all_qc = [], []
for index, folder in enumerate(folders, start=1):
    timepoint = parse_timepoint(folder.name)
    mask = read_label_mask(folder / NUCLEAR_FILTERED_MASK)
    all_cell_mask = read_label_mask(folder / CORRECTED_CELL_MASK)
    protein = np.asarray(
        tifffile.imread(folder / PROTEIN_ROLE["image_file"]), dtype=float,
    )
    if protein.ndim not in (2, 3):
        raise ValueError(
            f"Protein TIFF must be Y/X or Z/Y/X: {folder.name}, {protein.shape}"
        )
    protein_sum = protein if protein.ndim == 2 else protein.sum(axis=0)
    protein_max = protein if protein.ndim == 2 else protein.max(axis=0)
    if protein_sum.shape != mask.shape or all_cell_mask.shape != mask.shape:
        raise ValueError(f"Protein/mask shape mismatch in {folder.name}")
    if not np.isfinite(protein_sum).all():
        raise ValueError(f"Non-finite protein intensities in {folder.name}")

    cell_region = all_cell_mask > 0
    excluded = (
        binary_dilation(cell_region, iterations=BACKGROUND_EXCLUSION_PX)
        if BACKGROUND_EXCLUSION_PX > 0 else cell_region
    )
    background_region = ~excluded
    background_fallback = False
    if not np.any(background_region):
        background_region = ~cell_region
        background_fallback = True
    if not np.any(background_region):
        raise ValueError(f"No background pixels in {folder.name}; inspect segmentation")
    background_mean = float(protein_sum[background_region].mean())
    background_sd = float(protein_sum[background_region].std())

    spots = mrna_spots(load_stage04(folder))
    assigned = assign_spots(spots, mask)
    assigned_ids, assigned_counts = np.unique(assigned[assigned > 0], return_counts=True)
    counts = dict(zip(assigned_ids, assigned_counts))
    labels = np.unique(mask)
    labels = labels[labels > 0]

    for label in labels:
        region = mask == label
        area = int(region.sum())
        raw_mean = float(protein_sum[region].mean())
        raw_integrated = float(protein_sum[region].sum())
        all_cells.append({
            "Image_Folder": folder.name,
            "Timepoint_min": timepoint,
            "cell_id": int(label),
            "mRNA_Channel": MRNA_ROLE["channel"],
            "Protein_Channel": PROTEIN_ROLE["channel"],
            "Spot_Source": SPOT_SOURCE,
            "mrna_spot_count": int(counts.get(label, 0)),
            "Cell_Area_px": area,
            "Protein_Mean_Raw": raw_mean,
            "Protein_Integrated_Raw": raw_integrated,
            "Protein_Mean_BGCorrected": raw_mean - background_mean,
            "Protein_Integrated_BGCorrected": raw_integrated - background_mean * area,
            "Protein_MaxProjection_Mean": float(protein_max[region].mean()),
            "Protein_Background_Mean": background_mean,
            "Protein_Background_SD": background_sd,
            "Touches_Image_Border": touches_border(mask, label),
        })

    n_assigned = int(np.count_nonzero(assigned > 0))
    all_qc.append({
        "Image_Folder": folder.name,
        "Timepoint_min": timepoint,
        "mRNA_Channel": MRNA_ROLE["channel"],
        "Protein_Channel": PROTEIN_ROLE["channel"],
        "Spot_Source": SPOT_SOURCE,
        "N_Nuclear_Retained_Cells": len(labels),
        "N_mRNA_Spots_Decomposed": len(spots),
        "N_mRNA_Spots_Assigned": n_assigned,
        "N_mRNA_Spots_Unassigned": len(spots) - n_assigned,
        "Protein_Background_Mean": background_mean,
        "Protein_Background_SD": background_sd,
        "N_Background_Pixels": int(background_region.sum()),
        "Background_Exclusion_Fallback": background_fallback,
    })
    time_text = "missing" if np.isnan(timepoint) else f"{timepoint:g} min"
    print(
        f"[{index}/{len(folders)}] {folder.name}: "
        f"time={time_text}, {len(labels)} cells, {n_assigned}/{len(spots)} spots assigned"
    )

if not all_cells:
    raise RuntimeError("No nuclear-retained cells; inspect segmentation and overlap settings")
master_df = pd.DataFrame(all_cells).sort_values(
    ["Timepoint_min", "Image_Folder", "cell_id"], na_position="last",
)
qc_df = pd.DataFrame(all_qc).sort_values(
    ["Timepoint_min", "Image_Folder"], na_position="last",
)
master_df.to_csv(MASTER_CSV, index=False)
qc_df.to_csv(IMAGE_QC_CSV, index=False)
print(f"Saved {len(master_df):,} cells: {MASTER_CSV}")
print(f"Saved {len(qc_df):,} image-QC rows: {IMAGE_QC_CSV}")

In [ ]:
# %% 7. Separate QC plots for each image field
# Uses the combined per-cell CSV from the original Stage 6.
# No target, strain, timepoint, or additional configuration is required.

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PLOT_DIR = OUTPUT_ROOT / "image_field_qc_plots"

if not MASTER_CSV.is_file():
    raise FileNotFoundError(
        f"Combined per-cell CSV not found: {MASTER_CSV}. "
        "Run your original Stage 6 without target parsing first."
    )

df = pd.read_csv(MASTER_CSV)
if df.empty:
    raise RuntimeError("No cells available for plotting")

metrics = [
    ("mrna_spot_count", "Gaussian-decomposed mRNA spots per cell"),
    ("Cell_Area_px", "Cell area (pixels)"),
    (
        "Protein_Mean_BGCorrected",
        "Protein-associated mean fluorescence, background-corrected",
    ),
    (
        "Protein_Integrated_BGCorrected",
        "Protein-associated integrated fluorescence, background-corrected",
    ),
]

required_columns = ["Image_Folder"] + [metric for metric, _ in metrics]
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f"Missing CSV columns: {missing_columns}")
if df["Image_Folder"].isna().any():
    raise ValueError("Missing Image_Folder values in the CSV")

PLOT_DIR.mkdir(parents=True, exist_ok=True)
saved_paths = []

for index, (folder_name, image_df) in enumerate(
    df.groupby("Image_Folder", sort=True), start=1
):
    fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)

    for ax, (metric, ylabel) in zip(axes.ravel(), metrics):
        values = pd.to_numeric(image_df[metric], errors="coerce").to_numpy(dtype=float)
        values = values[np.isfinite(values)]
        if len(values) == 0:
            ax.text(
                0.5, 0.5, "No finite measurements",
                ha="center", va="center", transform=ax.transAxes,
            )
        else:
            # A density estimate is only meaningful for nonconstant measurements.
            if len(values) >= 2 and np.ptp(values) > 0:
                sns.violinplot(
                    y=values, color="#b7d7e8", cut=0,
                    inner="quartile", ax=ax,
                )
            sns.stripplot(
                y=values, color="#174a7e", alpha=0.35,
                size=2.2, jitter=0.18, ax=ax,
            )
        ax.set(xlabel="", ylabel=ylabel, title=metric)
        ax.set_xticks([])

    fig.suptitle(
        f"{folder_name}\nPer-cell measurements ({len(image_df):,} cells)",
        fontsize=14,
    )
    safe_name = re.sub(r"[^A-Za-z0-9._-]+", "_", str(folder_name)).strip("._-")
    if not safe_name:
        safe_name = "image_field"
    output_path = PLOT_DIR / f"{index:03d}_{safe_name}_qc_metrics.png"
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    saved_paths.append(output_path)
    print(f"Saved: {output_path}")

print(f"Saved {len(saved_paths)} separate image-field figures in {PLOT_DIR}")
